# B2-023 — Practice p17

*65 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

**Set:** C · **Type:** integrative · **Difficulty:** core · **Time budget:** 65 minutes  
**Concepts:** generative-adversarial-network

## Task

Train the unit's tiny MLP GAN on the seeded `mixture2d` data and certify its trace and its mode coverage.

**Data.** Load the unit loader with the supplied cell. Define `train_rows()` and `heldout_rows()` with no arguments; they return fresh float32 tensors `generative_data.train_tensor("mixture2d")` of shape `(256,2)` and `generative_data.heldout_tensor("mixture2d")` of shape `(64,2)`, built from the immutable `generative_data.TRAIN_IDS["mixture2d"]` and `generative_data.HELDOUT_IDS["mixture2d"]` in stored order. Held-out rows never enter a training call, a `backward()`, an optimizer step, or a hyperparameter choice. The clean training rows reach the model **only** through the discriminator's `forward`, exactly as returned (no normalization, noise, shuffling, or concatenation with other rows).

**Model and helpers.** `make_generator()` returns `nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 2))` and `make_discriminator()` returns `nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1))` (default initialization; the discriminator outputs logits and has no final activation). Implement `discriminator_loss`, `generator_loss`, `d_step`, `g_step`, and `gan_step` with the p07 contract:
`discriminator_loss(real_logits, fake_logits)` returns `F.binary_cross_entropy_with_logits(real_logits, torch.ones_like(real_logits)) + F.binary_cross_entropy_with_logits(fake_logits, torch.zeros_like(fake_logits))` (sum of two batch means) and `generator_loss(fake_logits)` returns `F.binary_cross_entropy_with_logits(fake_logits, torch.ones_like(fake_logits))` (non-saturating), both with the p06 contract.
`d_step(G, D, opt_d, real, z)` runs `opt_d.zero_grad(set_to_none=True)`, computes `real_logits = D(real)` and then `fake_logits = D(G(z).detach())`, forms `loss = discriminator_loss(real_logits, fake_logits)`, calls `loss.backward()` and `opt_d.step()`, and returns `loss.item()`.
`g_step(G, D, opt_g, z)` runs `opt_g.zero_grad(set_to_none=True)`, forms `loss = generator_loss(D(G(z)))`, calls `loss.backward()` and `opt_g.step()`, and returns `loss.item()`.
`gan_step(G, D, opt_g, opt_d, real, z)` calls `d_step(G, D, opt_d, real, z)` and then `g_step(G, D, opt_g, z)`, and returns the pair `(d_loss, g_loss)` of Python floats.
Implement `mode_coverage(samples, centers, radius)` with the p08 contract.

**Training.** Call `torch.manual_seed(SEED)`, then `G = make_generator()`, then `D = make_discriminator()`; `opt_g = torch.optim.Adam(G.parameters(), lr=5e-4, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)`, `opt_d = torch.optim.Adam(D.parameters(), lr=2e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)`. Write `train_gan(G, D, opt_g, opt_d, batch)` that creates `noise_generator = torch.Generator().manual_seed(SEED)` once and then performs **exactly 600** steps; each step draws `z = torch.randn(batch.shape[0], 2, generator=noise_generator)` and calls `gan_step(G, D, opt_g, opt_d, batch, z)`. It returns the list of 600 `(d_loss, g_loss)` pairs. Call it as `trace = train_gan(G, D, opt_g, opt_d, train_rows())`.

**Evaluation.** Under `torch.no_grad()`: `samples = G(torch.randn(512, 2, generator=torch.Generator().manual_seed(SEED + 2)))`; `counts, covered = mode_coverage(samples, generative_data.mixture_centers(), 0.6)`; `p_train = torch.sigmoid(D(train_rows())).mean()` and `p_held = torch.sigmoid(D(heldout_rows())).mean()`.

**Certify.**
(A) `len(trace) == 600`, and every entry is a pair of finite Python floats.
(B) **Discriminator band:** the mean of the last 50 discriminator losses lies within `0.95` of $2\log2$.
(C) **Generator bound:** every generator loss is finite and the largest is at most `5.0`.
(D) **Coverage:** `covered >= 2` with the p08 threshold $\lceil512/32\rceil=16$.
(E) **No memorization signal:** `abs(p_train - p_held) <= 0.10`.
(F) At least one parameter of each network changed, and `samples` has shape `(512,2)` and dtype float32.
(G) **Split certificate at the clean-row seam.** For the duration of the training call only, wrap `D.forward` so that it records every row it receives (restore the original afterwards; held-out evaluation later legitimately uses the same seam). Then assert that no recorded row hashes, with `generative_data.row_sha256`, into the held-out part of `generative_data.ROW_SHA256["mixture2d"]` (the hashes at the positions in `generative_data.HELDOUT_IDS["mixture2d"]`), and that the recorded hashes include every training row's hash.
These thresholds were set from the frozen-seed reference run with margin; they are not a priori guarantees.

**Interpretation.** In four sentences: report your counts vector; explain why (B) is consistent with a good generator but cannot certify one; explain why "at least 2 of 4 modes" is the honest bar for this protocol and what it says about mode collapse; and say why (E) compares training and held-out rows instead of training rows alone.

**Required answer-check assertions.** Every item (A)–(G), and the shapes `(256,2)` and `(64,2)` of the two splits.

In [ ]:
import importlib.util
import math
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261015
LOADER_CANDIDATES = (
    Path("../data/generative_data.py"),
    Path("units/B2-023-generative-models-diffusion/data/generative_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("generative_data", loader_path)
generative_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(generative_data)
torch.set_num_threads(1)

In [ ]:
# Write the required implementation, probes, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, shape/dtype probe, training certificate, or audit. Use only the permitted literal data and the seeded unit generator, keep train and held-out roles separate, and end coding work with an answer-check section of executable assertions.